# 2. Create the AI Search index
Use the original claims-evidence fields: one row per evidence chunk, one contextual content vector, and source metadata. Use a separate workshop index per participant. Creating an existing index fails with the SDK's error; nothing is replaced.

Non-spark environment: use this repository's `.venv` and root `.env`. This notebook does not extract documents, generate embeddings, or upload records.

In [ ]:
import os
from dotenv import load_dotenv, find_dotenv
from azure.identity import DefaultAzureCredential
from azure.search.documents.indexes import SearchIndexClient
from azure.search.documents.indexes.models import SearchField, SearchFieldDataType, SearchIndex, VectorSearch, HnswAlgorithmConfiguration, HnswParameters, VectorSearchProfile, SemanticConfiguration, SemanticSearch, SemanticPrioritizedFields, SemanticField
import pandas as pd
from IPython.display import display

load_dotenv(find_dotenv(usecwd=True), override=True)
credential = DefaultAzureCredential()

index_name = os.getenv('AZURE_SEARCH_CU_INDEX_NAME', 'claims-cu-v1')
azure_openai_model_dimensions = int(os.getenv('AZURE_OPENAI_EMBEDDING_DIMENSIONS', '3072'))
pd.set_option('display.max_rows', 5)
pd.set_option('display.max_columns', 6)
pd.set_option('display.max_colwidth', 80)
print('Index:', index_name, '| Vector dimensions:', azure_openai_model_dimensions)

## Step 1: Define the same 18 fields
These match the CU preprocessing notebook. Title/header/content/filename support text retrieval; native provenance supports citations. Full vectors are hidden from search responses.

In [ ]:
fields = [
    SearchField(name='id', type=SearchFieldDataType.String, key=True, filterable=True),
    SearchField(name='claim_id', type=SearchFieldDataType.String, filterable=True),
    SearchField(name='document_hash', type=SearchFieldDataType.String, filterable=True),
    SearchField(name='chunk_id', type=SearchFieldDataType.Int32, sortable=True),
    SearchField(name='title', type=SearchFieldDataType.String, searchable=True),
    SearchField(name='header', type=SearchFieldDataType.String, searchable=True),
    SearchField(name='content', type=SearchFieldDataType.String, searchable=True),
    SearchField(name='source_type', type=SearchFieldDataType.String, filterable=True, facetable=True),
    SearchField(name='content_kind', type=SearchFieldDataType.String, filterable=True, facetable=True),
    SearchField(name='filename', type=SearchFieldDataType.String, searchable=True, filterable=True),
    SearchField(name='source_path', type=SearchFieldDataType.String),
    SearchField(name='source_files', type=SearchFieldDataType.Collection(SearchFieldDataType.String)),
    SearchField(name='parent_emails', type=SearchFieldDataType.Collection(SearchFieldDataType.String), filterable=True),
    SearchField(name='figure_id', type=SearchFieldDataType.String, filterable=True),
    SearchField(name='source_region', type=SearchFieldDataType.String),
    SearchField(name='analyzer_id', type=SearchFieldDataType.String, filterable=True, facetable=True),
    SearchField(name='analyzer_api_version', type=SearchFieldDataType.String, filterable=True),
    SearchField(name='content_vector', type=SearchFieldDataType.Collection(SearchFieldDataType.Single), searchable=True, hidden=True, vector_search_dimensions=azure_openai_model_dimensions, vector_search_profile_name='claims-content-profile'),
]
display(pd.DataFrame([{'name': field.name, 'type': field.type, 'filterable': bool(field.filterable), 'searchable': bool(field.searchable), 'dimensions': field.vector_search_dimensions} for field in fields]).head(5))
print('Fields:', len(fields), '|', ', '.join(field.name for field in fields))

## Step 2: Vector and semantic configuration
Use HNSW/cosine, one contextual content vector, and the original `claims-semantic` configuration.

In [ ]:
vector_search = VectorSearch(algorithms=[HnswAlgorithmConfiguration(name='claims-hnsw', parameters=HnswParameters(metric='cosine'))], profiles=[VectorSearchProfile(name='claims-content-profile', algorithm_configuration_name='claims-hnsw')])
semantic_config = SemanticConfiguration(name='claims-semantic', prioritized_fields=SemanticPrioritizedFields(title_field=SemanticField(field_name='title'), content_fields=[SemanticField(field_name='content'), SemanticField(field_name='header')]))
semantic_search = SemanticSearch(configurations=[semantic_config], default_configuration_name='claims-semantic')
index = SearchIndex(name=index_name, fields=fields, vector_search=vector_search, semantic_search=semantic_search)
print('Vector: HNSW/cosine | Semantic: claims-semantic')

## Step 3: Create the index (Azure write)
Run once for a new index name. If it already exists, the SDK reports that error directly. Skip this cell when reusing your workshop index; this notebook never deletes or updates an index.

In [ ]:
with SearchIndexClient(endpoint=os.environ['AZURE_SEARCH_SERVICE_ENDPOINT'], credential=credential) as index_client:
    index_client.create_index(index)
print(index_name, 'created')
credential.close()
print('Next: run 03_populate_index.ipynb.')